# XAI TCNER Project

## Train Base Model

In [ ]:
import pandas as pd
import numpy as np
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ----------------------------
# Load spaCy and define preprocessing
# ----------------------------
nlp = spacy.load("en_core_web_sm")

def preprocess(text):
    doc = nlp(text.lower())
    tokens = [
        token.lemma_ for token in doc
        if not token.is_stop and not token.is_punct and len(token) > 2
    ]
    return " ".join(tokens)

# ----------------------------
# Load data
# ----------------------------
train_df = pd.read_csv("DBLPTrainset.txt", sep="\t", header=None, names=["Label", "Title"])
test_df = pd.read_csv("DBLPTestset.txt", sep="\t", header=None, names=["Title"])
test_truth = pd.read_csv("DBLPTestGroundTruth.txt", sep="\t", header=None, names=["Label"])
y_test = test_truth["Label"]

# ----------------------------
# Preprocess titles
# ----------------------------
train_df["Cleaned_Title"] = train_df["Title"].apply(preprocess)
test_df["Cleaned_Title"] = test_df["Title"].apply(preprocess)

y_train = train_df["Label"]
y_test = test_truth["Label"]

print("Class Distribution in Training Set:")
print(y_train.value_counts())
print("\nClass Distribution in Test Set:")
print(y_test.value_counts())

# ----------------------------
# Data Quality Checks
# ----------------------------
train_dup_titles = train_df.duplicated(subset="Title").sum()
test_dup_titles = test_df.duplicated(subset="Title").sum()
unique_train_dup_titles = train_df[train_df.duplicated(subset="Title", keep=False)]["Title"].nunique()
train_full_row_dups = train_df.duplicated().sum()
test_full_row_dups = test_df.duplicated().sum()

print("Train duplicate titles:", train_dup_titles)
print("Unique duplicate titles in train set:", unique_train_dup_titles)
print("Train full-row duplicates:", train_full_row_dups)
print("Test duplicate titles:", test_dup_titles)
print("Test full-row duplicates:", test_full_row_dups)

train_df = train_df.drop_duplicates(subset="Title", keep="first")

print("\nMissing values in training set:")
print(train_df.isnull().sum())
print("\nMissing values in test set:")
print(test_df.isnull().sum())
print("\nMissing values in test ground truth:")
print(test_truth.isnull().sum())

# ----------------------------
# Outlier Detection (Title Length)
# ----------------------------
train_df["title_char_len"] = train_df["Title"].apply(len)
test_df["title_char_len"] = test_df["Title"].apply(len)

print("\nOutlier Check - Title Lengths:")
print("Training set → Min:", train_df["title_char_len"].min(), "| Max:", train_df["title_char_len"].max())
print("Test set     → Min:", test_df["title_char_len"].min(), "| Max:", test_df["title_char_len"].max())

# ----------------------------
# TF-IDF vectorization
# ----------------------------
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words='english',
    min_df=3,
    max_df=0.85
)

X_train = vectorizer.fit_transform(train_df["Cleaned_Title"])
X_test = vectorizer.transform(test_df["Cleaned_Title"])
y_train = train_df["Label"]

# ----------------------------
# Train model: Logistic Regression 
# ----------------------------
print("\nTraining LogisticRegression...")
model = LogisticRegression(max_iter=1000, solver='liblinear', multi_class='ovr')
model.fit(X_train, y_train)

# Predict on test set
y_pred = model.predict(X_test)

# Evaluation
print("\n=== FINAL TEST SET EVALUATION ===")
print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

cm = confusion_matrix(y_test, y_pred)

In [ ]:
from scipy.stats import entropy
import matplotlib.pyplot as plt
import numpy as np

probs = model.predict_proba(X_test)
entropies = entropy(probs, axis=1)

plt.figure(figsize=(8, 4))
plt.hist(entropies, bins=30, color='skyblue', edgecolor='black')
plt.title("Distribution of Prediction Entropies")
plt.xlabel("Entropy (Uncertainty)")
plt.ylabel("Number of Samples")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

y_pred = model.predict(X_test)
misclassified_mask = y_pred != y_test.values
misclassified_indices = np.where(misclassified_mask)[0]
misclassified_entropies = entropies[misclassified_indices]

sorted_idx = misclassified_indices[np.argsort(misclassified_entropies)]

print("\n=== Top 5 Overconfident Misclassified Samples ===")
for i in sorted_idx[:5]:
    print(f"\nIndex: {i}")
    print(f"Entropy       : {entropies[i]:.4f}")
    print(f"Title         : {test_df.iloc[i]['Title']}")
    print(f"True Label    : {y_test.iloc[i]}")
    print(f"Predicted     : {y_pred[i]}")